# 証券営業インテリジェンス ハンズオン
## Part 5: Cortex Search（セマンティック検索）

このノートブックでは、**Cortex Search** を使ってニュース・商品説明書・アナリストレポートを意味的に検索します。

### このパートで体験できること

1. **Cortex Search Service の作成**: 3種類のデータにインデックスを作成
2. **キーワード検索 vs セマンティック検索**: 従来の検索との圧倒的な違いを体験
3. **フィルター検索**: 感情・銘柄コードなどで絞り込み
4. **高度な検索機能**: Numeric Boosts（重要度スコア）・Time Decays（時間減衰）

### 🚀 体験ポイント

> **「証券担保ローン」と検索しなくても、**
> **「株を売らずにお金を借りたい」で同じ商品説明書がヒットする！**
>
> 顧客からの相談内容をそのままクエリにできるため、
> 担当者が専門用語を知らなくても適切な情報を引き出せます。

### 前提条件
- `setup.sql` 実行済み（ニュース・ローン書類・アナリストレポートのデータ投入済み）
- `part1_security.ipynb` 実行済み（`COMPUTE_WH` は `setup.sql` で作成済み）

> ⏱️ **このパートの目安時間: 45分**

In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE COMPUTE_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. 対象データの確認

Cortex Search Service を作成する前に、**インデックス対象のデータ**を確認します。
データの構造を理解することで、どの列を検索対象にするかを設計できます。

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_count
-- 各テーブルのレコード数を確認
SELECT 'NEWS_ARTICLES' AS "テーブル名", COUNT(*) AS "レコード数",
       '50件のマーケットニュース' AS "説明" FROM NEWS_ARTICLES
UNION ALL
SELECT 'LOAN_PRODUCT_DOCS', COUNT(*), 'ローン商品説明書（複数セクションに分割済み）' FROM LOAN_PRODUCT_DOCS
UNION ALL
SELECT 'ANALYST_REPORTS', COUNT(*), 'アナリストレポート（30社分）' FROM ANALYST_REPORTS
ORDER BY "テーブル名";

In [ ]:
%%sql -r result_news_check
-- NEWS_ARTICLES のデータ確認
SELECT
    NEWS_ID,
    PUBLISH_DATE AS "発行日",
    CATEGORY AS "カテゴリ",
    LEFT(TITLE, 60) AS "タイトル",
    SENTIMENT AS "感情",
    IMPORTANCE AS "重要度",
    RELATED_SECURITIES AS "関連銘柄"
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 5;

In [ ]:
%%sql -r result_loan_check
-- LOAN_PRODUCT_DOCS のデータ確認
SELECT
    DOC_ID,
    TITLE AS "ドキュメント名",
    SECTION AS "セクション",
    LEFT(CONTENT, 200) AS "本文（先頭200字）"
FROM LOAN_PRODUCT_DOCS
LIMIT 5;

In [ ]:
%%sql -r result_analyst_check
-- ANALYST_REPORTS のデータ確認（列の意味を確認）
SELECT
    REPORT_ID,
    PUBLISH_DATE AS "発行日",
    SECURITY_CODE AS "銘柄コード",
    SECURITY_NAME AS "銘柄名",
    ANALYST_NAME AS "アナリスト名",
    RATING AS "投資判断",
    TARGET_PRICE AS "目標株価"
FROM ANALYST_REPORTS
ORDER BY PUBLISH_DATE DESC
LIMIT 5;

## 2. Cortex Search Service の作成

各データソースに対して Cortex Search Service を作成します。

### Cortex Search Service の仕組み

```
CREATE CORTEX SEARCH SERVICE サービス名
ON 検索対象列（テキスト）       ← ここをベクトル化してインデックス化
ATTRIBUTES 絞り込み可能な列    ← フィルタリングに使える列
WAREHOUSE = ウェアハウス名     ← インデックス構築に使用
TARGET_LAG = '更新間隔'
AS ( SELECT ... FROM テーブル );
```

> **Note**: インデックス構築には数分かかります。完了を待たずに次の手順に進めます。
> （`SHOW CORTEX SEARCH SERVICES` で状態を確認できます）

> ⏱️ **目安: 10分**（コマンド実行は即時、インデックス構築は並行して進む）

### 2-1. ニュース記事検索サービス

マーケットニュース50件を検索対象とします。
`IMPORTANCE`（重要度1-5）と `PUBLISH_DATETIME`（発行日時）を含めて、
後の高度な検索機能（Numeric Boosts / Time Decays）で使用します。

In [ ]:
%%sql -r result_cs1
-- Cortex Search Service 1: マーケットニュース
CREATE OR REPLACE CORTEX SEARCH SERVICE NEWS_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES CATEGORY, TITLE, RELATED_SECURITIES, PUBLISH_DATE, SENTIMENT
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        NEWS_ID,
        PUBLISH_DATE,
        PUBLISH_DATETIME,
        CATEGORY,
        TITLE,
        CONTENT,
        SUMMARY,
        RELATED_SECURITIES,
        SENTIMENT,
        IMPORTANCE
    FROM NEWS_ARTICLES
);
SELECT 'NEWS_SEARCH_SERVICE: 作成コマンド実行完了（インデックス構築中）' AS STATUS;

### 2-2. ローン商品説明書検索サービス

証券担保ローン・教育資金贈与信託などの商品説明書を検索対象とします。

> **💡 これが今回の核心部分です！**
> このサービスを使うと、「証券担保ローン」という専門用語を知らなくても
> 「株を売らずに資金調達したい」というキーワードで商品説明書を発見できます。

In [ ]:
%%sql -r result_cs2
-- Cortex Search Service 2: ローン商品説明書
CREATE OR REPLACE CORTEX SEARCH SERVICE LOAN_DOCS_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES PRODUCT_ID, DOC_TYPE, SECTION, TITLE
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        DOC_ID,
        PRODUCT_ID,
        DOC_TYPE,
        SECTION,
        TITLE,
        CONTENT
    FROM LOAN_PRODUCT_DOCS
);
SELECT 'LOAN_DOCS_SEARCH_SERVICE: 作成コマンド実行完了（インデックス構築中）' AS STATUS;

### 2-3. アナリストレポート検索サービス

アナリストレポートの投資論拠を検索対象とします。
銘柄コード・投資判断・アナリスト名でフィルタリングが可能です。

In [ ]:
%%sql -r result_cs3
-- Cortex Search Service 3: アナリストレポート
CREATE OR REPLACE CORTEX SEARCH SERVICE ANALYST_REPORT_SEARCH_SERVICE
ON CONTENT
ATTRIBUTES SECURITY_CODE, SECURITY_NAME, RATING, PUBLISH_DATE, ANALYST_NAME
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        REPORT_ID,
        PUBLISH_DATE,
        SECURITY_CODE,
        SECURITY_NAME,
        ANALYST_NAME,
        RATING,
        TARGET_PRICE,
        REPORT_TITLE,
        EXECUTIVE_SUMMARY,
        INVESTMENT_THESIS AS CONTENT,
        KEY_RISKS
    FROM ANALYST_REPORTS
);
SELECT 'ANALYST_REPORT_SEARCH_SERVICE: 作成コマンド実行完了（インデックス構築中）' AS STATUS;

In [ ]:
%%sql -r result_show_cs
-- 作成した Cortex Search Service の状態確認
-- READY になるまでインデックス構築中（数分かかる場合あり）
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

## 3. キーワード検索 vs セマンティック検索

### 従来のキーワード検索の限界

SQL の `LIKE` 句による従来のキーワード検索は、**文字の完全一致** しか検索できません。
顧客が「株を担保にお金を借りたい」と言っても、「証券担保ローン」というキーワードがないと見つかりません。

### Cortex Search の革新

**意味（セマンティクス）を理解**した検索ができます。
「株を売らずに資金調達」→「証券担保ローン」を発見！

> ⏱️ **目安: 10分**

> **💡 比較の準備**: 同じ質問をキーワード検索とセマンティック検索で試して、
> 結果の違いを確認してください！

In [ ]:
%%sql -r result_keyword_search
-- 【従来のキーワード検索（LIKE句）】
-- 「株を売らずに資金を調達したい」でLIKE検索 → ヒットしない！
SELECT
    DOC_ID,
    TITLE,
    SECTION,
    LEFT(CONTENT, 200) AS "本文"
FROM LOAN_PRODUCT_DOCS
WHERE CONTENT LIKE '%株を売らずに%'
   OR CONTENT LIKE '%資金を調達%'
   OR TITLE LIKE '%株を売らずに%';
-- 結果: 0件（専門用語「証券担保ローン」がクエリにないため）

In [ ]:
%%sql -r result_semantic_search
-- 【Cortex Search（セマンティック検索）】
-- まったく同じ質問でも、意味を理解してヒットする！
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SECTION::STRING                AS "セクション",
    LEFT(f.value:CONTENT::STRING, 300)     AS "本文（先頭300字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "株を売らずに資金を調達したい",
            "columns": ["DOC_ID", "TITLE", "SECTION", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;
-- 結果: 「証券担保ローン」の説明書が上位にヒット！

## 4. 基本的なセマンティック検索

### 検索結果の読み方

| 列名 | 意味 |
|---|---|
| `類似度スコア` | 0〜1の類似度（1に近いほど関連度が高い） |
| `TITLE` | ドキュメントのタイトル |
| `SECTION` | ドキュメントのセクション名 |
| `CONTENT` | 検索対象のテキスト（一部） |

> ⏱️ **目安: 10分**

### 4-1. ローン商品説明書の意味検索

In [ ]:
%%sql -r result_search_loan1
-- 「株を売らずに資金を調達したい」- 証券担保ローンを発見
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:DOC_ID::STRING                 AS "ドキュメントID",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SECTION::STRING                AS "セクション",
    LEFT(f.value:CONTENT::STRING, 200)     AS "本文（先頭200字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "株を売らずに資金を調達したい",
            "columns": ["DOC_ID", "TITLE", "SECTION", "CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
%%sql -r result_search_loan2
-- 「孫への教育費用を節税しながら渡す方法」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SECTION::STRING                AS "セクション",
    LEFT(f.value:CONTENT::STRING, 250)     AS "本文（先頭250字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{
            "query": "孫への教育費用を節税しながら渡す方法",
            "columns": ["TITLE", "SECTION", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 4-2. ニュース記事の検索

In [ ]:
%%sql -r result_search_news1
-- 「日本銀行の金融政策と株式市場への影響」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:PUBLISH_DATE::STRING           AS "発行日",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SENTIMENT::STRING              AS "センチメント",
    LEFT(f.value:CONTENT::STRING, 200)     AS "本文（先頭200字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "日本銀行の金融政策と株式市場への影響",
            "columns": ["PUBLISH_DATE", "TITLE", "CONTENT", "SENTIMENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 4-3. アナリストレポートの検索

In [ ]:
%%sql -r result_search_analyst1
-- 「EV シフトとトヨタの競争優位性」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:SECURITY_NAME::STRING          AS "銘柄名",
    f.value:ANALYST_NAME::STRING           AS "アナリスト",
    f.value:RATING::STRING                 AS "投資判断",
    f.value:TARGET_PRICE::STRING           AS "目標株価",
    LEFT(f.value:CONTENT::STRING, 250)     AS "投資根拠（先頭250字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE',
        '{
            "query": "EV シフトとトヨタの競争優位性",
            "columns": ["SECURITY_NAME", "ANALYST_NAME", "RATING", "TARGET_PRICE", "CONTENT"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 5. フィルター付き検索

`ATTRIBUTES` で指定した列を使って、**検索結果を絞り込む**ことができます。

**フィルターの書式:**
```json
"filter": {"@eq": {"列名": "値"}}       // 完全一致
"filter": {"@contains": {"列名": "値"}} // 部分一致（文字列）
```

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_filter_positive
-- ポジティブなニュースのみ検索（感情フィルター）
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:PUBLISH_DATE::STRING           AS "発行日",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SENTIMENT::STRING              AS "センチメント"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "トヨタ自動車の業績と今後の見通し",
            "columns": ["PUBLISH_DATE", "TITLE", "SENTIMENT", "CONTENT"],
            "filter": {"@eq": {"SENTIMENT": "ポジティブ"}},
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
%%sql -r result_filter_negative
-- ネガティブなニュースのみ検索（ポートフォリオリスクアラート）
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:PUBLISH_DATE::STRING           AS "発行日",
    f.value:TITLE::STRING                  AS "タイトル",
    f.value:SENTIMENT::STRING              AS "センチメント",
    LEFT(f.value:CONTENT::STRING, 150)     AS "本文概要"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "株価下落リスク 投資家への影響",
            "columns": ["PUBLISH_DATE", "TITLE", "SENTIMENT", "CONTENT"],
            "filter": {"@eq": {"SENTIMENT": "ネガティブ"}},
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
%%sql -r result_filter_security
-- 特定銘柄（トヨタ 7203）のアナリストレポートをフィルタ検索
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:ANALYST_NAME::STRING           AS "アナリスト",
    f.value:RATING::STRING                 AS "投資判断",
    f.value:TARGET_PRICE::STRING           AS "目標株価",
    LEFT(f.value:CONTENT::STRING, 200)     AS "投資根拠"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE',
        '{
            "query": "ハイブリッド車の競争力と収益性",
            "columns": ["ANALYST_NAME", "RATING", "TARGET_PRICE", "CONTENT"],
            "filter": {"@eq": {"SECURITY_CODE": "7203"}},
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 6. 高度な検索機能

Cortex Search には、よりインテリジェントな検索を実現するための高度な機能があります。

| 機能 | 説明 | 証券営業での活用例 |
|---|---|---|
| **Numeric Boosts** | 数値列の値でスコアをブースト | 重要度の高いニュースを上位に |
| **Time Decays** | 時間が新しいほど高スコア | 最新のニュースを優先表示 |
| **組み合わせ** | フィルター + ブースト + 減衰を組み合わせ | 最新かつ重要なポジティブニュース |

> ⏱️ **目安: 10分**

### 6-1. Numeric Boosts（重要度スコアでブースト）

`IMPORTANCE`（重要度 1〜5）の値が高いニュースを上位に表示するよう調整します。

**パラメータ:**
```json
"numeric_boosts": [{"col": "IMPORTANCE", "factor": 2.0}]
```
- `factor`: ブーストの強さ（大きいほど数値の影響が強くなる）

In [ ]:
%%sql -r result_no_boost
-- 【比較用】ブーストなし: セマンティック検索のみ
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:IMPORTANCE::INT          AS "重要度",
    f.value:TITLE::STRING            AS "タイトル"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "日本株式市場の見通し",
            "columns": ["TITLE", "CONTENT", "IMPORTANCE", "PUBLISH_DATE"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 1 DESC;

In [ ]:
%%sql -r result_with_boost
-- Numeric Boosts: 重要度が高いニュースをスコアブースト
-- 重要度5のニュースが上位に来やすくなる
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "ブースト後スコア",
    f.value:IMPORTANCE::INT          AS "重要度",
    f.value:TITLE::STRING            AS "タイトル"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "日本株式市場の見通し",
            "columns": ["TITLE", "CONTENT", "IMPORTANCE", "PUBLISH_DATE"],
            "numeric_boosts": [{"col": "IMPORTANCE", "factor": 2.0}],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 1 DESC;

### 6-2. Time Decays（最新ニュースを優先）

最新のニュースほど高いスコアになるように調整します。
古いニュースより新しいニュースを優先したい場合に使います。

**パラメータ:**
```json
"time_decays": [{"col": "PUBLISH_DATETIME", "target_time": "YYYY-MM-DDThh:mm:ss", "limit_hours": N}]
```
- `target_time`: 最もスコアが高くなる時刻（通常は最新の日付）
- `limit_hours`: この時間より古いデータは急速にスコアが低下

In [ ]:
%%sql -r result_no_decay
-- 【比較用】時間減衰なし: セマンティック検索のみ
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "スコア",
    f.value:PUBLISH_DATE::STRING            AS "発行日",
    f.value:TITLE::STRING                   AS "タイトル"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "金利上昇の債券市場への影響",
            "columns": ["TITLE", "CONTENT", "PUBLISH_DATE", "PUBLISH_DATETIME"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 1 DESC;

In [ ]:
%%sql -r result_with_decay
-- Time Decays: 最新ニュースを優先（直近1年のニュースを優先）
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "時間減衰後スコア",
    f.value:PUBLISH_DATE::STRING            AS "発行日",
    f.value:TITLE::STRING                   AS "タイトル"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "金利上昇の債券市場への影響",
            "columns": ["TITLE", "CONTENT", "PUBLISH_DATE", "PUBLISH_DATETIME"],
            "time_decays": [{"col": "PUBLISH_DATETIME", "target_time": "2025-03-01T00:00:00", "limit_hours": 8760}],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 1 DESC;

### 6-3. 組み合わせ検索（フィルター + ブースト + 減衰）

3つの機能を組み合わせることで、より精度の高い検索が可能です。

**例**: ポジティブなニュースの中で、重要度が高く、最新のニュースを検索

In [ ]:
%%sql -r result_combined_search
-- 組み合わせ検索: フィルター + Numeric Boosts + Time Decays
-- ポジティブニュース × 重要度ブースト × 最新優先
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "最終スコア",
    f.value:PUBLISH_DATE::STRING            AS "発行日",
    f.value:IMPORTANCE::INT                 AS "重要度",
    f.value:SENTIMENT::STRING              AS "センチメント",
    f.value:TITLE::STRING                   AS "タイトル"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{
            "query": "株式投資の好機 企業業績改善",
            "columns": ["TITLE", "CONTENT", "PUBLISH_DATE", "PUBLISH_DATETIME", "IMPORTANCE", "SENTIMENT"],
            "filter": {"@eq": {"SENTIMENT": "ポジティブ"}},
            "numeric_boosts": [{"col": "IMPORTANCE", "factor": 1.5}],
            "time_decays": [{"col": "PUBLISH_DATETIME", "target_time": "2025-03-01T00:00:00", "limit_hours": 4380}],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 1 DESC;

## 7. 山田様の相談への総合活用

「トヨタ株を売って孫の留学費用を捻出したい」という相談に対して、
3つの Cortex Search Service から関連情報を一括収集します。

> **💡 これが Cortex Agent（Part 4）への橋渡しになります！**
> Part 4 では、このような複数サービスへの並行検索を
> AIエージェントが自動で行ってくれます。

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_yamada_comprehensive
-- 山田様（C001）の相談に関連する情報を全サービスから収集
SELECT '【関連ニュース】' AS "種別",
    f.value:PUBLISH_DATE::STRING AS "日付",
    f.value:TITLE::STRING AS "タイトル",
    f.value:SENTIMENT::STRING AS "補足",
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "スコア"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE',
        '{"query":"トヨタ株 売却 税金 留学 教育費","columns":["PUBLISH_DATE","TITLE","SENTIMENT"],"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

UNION ALL

SELECT '【商品説明書】',
    NULL,
    f.value:TITLE::STRING,
    f.value:SECTION::STRING,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3)
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE',
        '{"query":"株を売らずに孫の教育費を捻出したい","columns":["TITLE","SECTION","CONTENT"],"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

UNION ALL

SELECT '【アナリストレポート】',
    f.value:PUBLISH_DATE::STRING,
    f.value:SECURITY_NAME::STRING || ' / ' || f.value:ANALYST_NAME::STRING AS TITLE,
    f.value:RATING::STRING,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3)
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE',
        '{"query":"トヨタ自動車 株価 今後の見通し","columns":["SECURITY_NAME","ANALYST_NAME","RATING","CONTENT"],"filter":{"@eq":{"SECURITY_CODE":"7203"}},"limit":3}'
    ) AS r
), LATERAL FLATTEN(input => PARSE_JSON(r):results) AS f

ORDER BY 5 DESC;

## まとめ

Part 3 完了！Cortex Search の豊富な機能を体験しました。

### 作成した Cortex Search Services

| サービス名 | 対象データ | 件数 | 検索対象列 |
|---|---|---|---|
| `NEWS_SEARCH_SERVICE` | マーケットニュース | 50件 | CONTENT |
| `LOAN_DOCS_SEARCH_SERVICE` | ローン商品説明書 | 複数セクション | CONTENT |
| `ANALYST_REPORT_SEARCH_SERVICE` | アナリストレポート | 30件 | INVESTMENT_THESIS |

### 体験した機能

| 機能 | 具体的な効果 |
|---|---|
| **キーワード検索 vs セマンティック** | 「証券担保ローン」を知らなくても発見できた |
| **フィルター検索** | ポジティブなニュースのみ、特定銘柄のみを絞り込み |
| **Numeric Boosts** | 重要度の高いニュースを上位表示 |
| **Time Decays** | 最新ニュースを優先 |
| **組み合わせ検索** | 3機能を組み合わせて精度向上 |

### 重要なポイント

1. **`CREATE CORTEX SEARCH SERVICE` の1行** でインデックスが自動構築・更新
2. **`SEARCH_PREVIEW` 関数** で任意のSQLから検索が可能
3. **フィルター・ブースト・減衰** の組み合わせで高精度な検索を実現

**次のステップ:** `part6_cortex_agent.ipynb` でこれらすべてを統合したエージェントを作成してください。